# 03 - Predictive Tests

This notebook tests whether SGI predicts future market stress.

## Steps:
1. Build feature and target datasets
2. Run walk-forward predictive tests
3. Compare feature sets (baseline vs SGI)
4. Analyze incremental value of SGI

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import load_experiment_config
from src.experiments import run_signal_pipeline
from src.targets import build_target_dataframe
from src.features import get_feature_columns
from src.predictive_models import (
    prepare_model_dataset, run_walk_forward_experiment,
    summarize_walk_forward_results, compute_incremental_r2
)
from src.paths import ensure_all_directories

ensure_all_directories()

## Run Signal Pipeline

In [ ]:
config = load_experiment_config('../configs/universe_sector_etfs.yaml')
signal_results = run_signal_pipeline(config)

features_df = signal_results['features_df']
prices = signal_results['prices']
returns = signal_results['returns']
cov_matrices = signal_results['cov_matrices']

## Build Target Variables

In [ ]:
targets_df = build_target_dataframe(prices, returns, cov_matrices, config)
print(f"Targets shape: {targets_df.shape}")
print(f"Target columns: {list(targets_df.columns)}")

## Define Feature Sets

In [ ]:
feature_sets = {
    'baseline_only': get_feature_columns('baseline_only', config),
    'sgi_only': get_feature_columns('sgi_only', config),
    'baseline_plus_sgi': get_feature_columns('baseline_plus_sgi', config),
}

for name, cols in feature_sets.items():
    print(f"{name}: {cols}")

## Run Walk-Forward Tests

In [ ]:
# Test on forward drawdown target
target_col = 'forward_drawdown_20d'

dataset = prepare_model_dataset(
    features_df, targets_df, target_col,
    get_feature_columns('full', config)
)

print(f"Dataset shape: {dataset.shape}")
print(f"Date range: {dataset.index.min()} to {dataset.index.max()}")

In [ ]:
results = run_walk_forward_experiment(
    dataset, 'target', feature_sets,
    model_type='linear', config=config
)

print(f"Number of folds: {results['n_splits']}")

In [ ]:
summary = summarize_walk_forward_results(results)
print("Walk-Forward Results Summary:")
summary

## Incremental Value Analysis

In [ ]:
incremental = compute_incremental_r2(results)
print("Incremental R² from adding SGI:")
for key, value in incremental.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

In [ ]:
# Visualize results
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(summary))
ax.bar(x, summary['mean_r2'])
ax.set_xticks(x)
ax.set_xticklabels(summary['feature_set'], rotation=45, ha='right')
ax.set_ylabel('Mean Out-of-Sample R²')
ax.set_title(f'Predictive Performance: {target_col}')
ax.axhline(y=0, color='gray', linestyle='--')

plt.tight_layout()
plt.show()

## Key Findings

1. Compare baseline_only vs baseline_plus_sgi R²
2. Positive incremental R² indicates SGI adds predictive value
3. Check consistency across folds